In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [36]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response

<Response [200]>

In [37]:
soup = BeautifulSoup(response.content)

In [38]:
books_grid = soup.find("ol", attrs = {"class": "row"})

In [39]:
books = books_grid.find_all("li", attrs = {"class": "col-xs-6 col-sm-4 col-md-3 col-lg-3"})

In [40]:
books[0].find_all("a")[1]["title"]

'A Light in the Attic'

In [41]:
books[0].find_all("a")[1]["href"]

'a-light-in-the-attic_1000/index.html'

In [42]:
books[0].find("p", attrs = {"class": "price_color"}).get_text().replace("£","")

'51.77'

In [43]:
def get_title(book): 
    title = book.find_all("a")[1]["title"]
    return title

def get_url(book):
    relative_url = book.find_all("a")[1]["href"]
    
    base_url = "https://books.toscrape.com/catalogue/"
    
    cleaned_url = relative_url.replace('../../../', '')
    return base_url + cleaned_url

def get_price(book):
    price = book.find("p", attrs = {"class": "price_color"}).get_text().replace("£","")
    return price

In [44]:
books_dict = {}
key = 0

for book in books:
    title = get_title(book)
    price = get_price(book)
    url = get_url(book)

    books_dict[key] = {"title": title,
                        "price": price,
                        "url": url}

    key +=1

books_df = pd.DataFrame.from_dict(books_dict, orient = "index")

In [46]:
books_df.head(3)

,title,price,url
0,A Light in the Attic,51.77,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,https://books.toscrape.com/catalogue/soumissio...


In [47]:
import requests
from bs4 import BeautifulSoup

def get_book_details(book_url):
    response = requests.get(book_url)
    soup = BeautifulSoup(response.content, "html.parser")

    # UPC
    upc = soup.find("th", string="UPC").find_next_sibling("td").text.strip()

    # Rating
    rating_tag = soup.find("p", class_="star-rating")
    rating_str = rating_tag["class"][1]  # e.g. "Four"
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    rating = rating_map.get(rating_str, 0)

    # Genre
    genre = soup.select("ul.breadcrumb li")[2].text.strip()

    # Availability
    availability = soup.find("p", class_="instock availability").text.strip()

    # Description
    desc_tag = soup.find("div", id="product_description")
    if desc_tag:
        description = desc_tag.find_next_sibling("p").text.strip()
    else:
        description = None

    return {
        "UPC": upc,
        "Rating": rating,
        "Genre": genre,
        "Availability": availability,
        "Description": description
    }

In [56]:
MAX_BOOKS = 1000
books_dict = {}
key = 0
page_num = 1

min_rating = 4
max_price = 20

while True:
    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(url)
    if response.status_code != 200:
        break  # no more pages

    soup = BeautifulSoup(response.content, "html.parser")
    books_grid = soup.find("ol", class_="row")
    books = books_grid.find_all("li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3")

    if not books:
        break

    for book in books:
        if key >= MAX_BOOKS:
            break  # STOP SCRAPING AFTER 10 BOOKS

        title = get_title(book)
        price = float(get_price(book))
        url = get_url(book)

        details = get_book_details(url)

        if price > max_price or details["Rating"] < min_rating:
          continue 

        books_dict[key] = {
            "Title": title,
            "Price (£)": price,
            "URL": url,
            "UPC": details["UPC"],
            "Rating": details["Rating"],
            "Genre": details["Genre"],
            "Availability": details["Availability"],
            "Description": details["Description"]
        }
        key += 1

    if key >= MAX_BOOKS:
        break  # stop after MAX reached

    page_num += 1

In [57]:
import pandas as pd

books_df = pd.DataFrame.from_dict(books_dict, orient='index')
print(books_df.head())

                                               Title  Price (£)  \
0                                        Set Me Free      17.46   
1  The Four Agreements: A Practical Guide to Pers...      17.66   
2                                     Sophie's World      15.94   
3            Untitled Collection: Sabbath Poems 2014      14.27   
4                                    This One Summer      19.49   

                                                 URL               UPC  \
0  https://books.toscrape.com/catalogue/set-me-fr...  ce6396b0f23f6ecc   
1  https://books.toscrape.com/catalogue/the-four-...  6258a1f6a6dcfe50   
2  https://books.toscrape.com/catalogue/sophies-w...  6be3beb0793a53e7   
3  https://books.toscrape.com/catalogue/untitled-...  657fe5ead67a7767   
4  https://books.toscrape.com/catalogue/this-one-...  51653ef291ab7ddc   

   Rating           Genre             Availability  \
0       5     Young Adult  In stock (19 available)   
1       5    Spirituality  In stock (18 avai

In [63]:
books_df.head(8)

,Title,Price (£),URL,UPC,Rating,Genre,Availability,Description
0,Set Me Free,17.46,https://books.toscrape.com/catalogue/set-me-fr...,ce6396b0f23f6ecc,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,The Four Agreements: A Practical Guide to Pers...,17.66,https://books.toscrape.com/catalogue/the-four-...,6258a1f6a6dcfe50,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,Sophie's World,15.94,https://books.toscrape.com/catalogue/sophies-w...,6be3beb0793a53e7,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,Untitled Collection: Sabbath Poems 2014,14.27,https://books.toscrape.com/catalogue/untitled-...,657fe5ead67a7767,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,This One Summer,19.49,https://books.toscrape.com/catalogue/this-one-...,51653ef291ab7ddc,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
5,Thirst,17.27,https://books.toscrape.com/catalogue/thirst_94...,709822d0b5bcb7f4,5,Fiction,In stock (16 available),"On a searing summer Friday, Eddie Chapman has ..."
6,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,https://books.toscrape.com/catalogue/princess-...,0fa6dceead7ce47a,5,Sequential Art,In stock (16 available),THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...
7,Princess Between Worlds (Wide-Awake Princess #5),13.34,https://books.toscrape.com/catalogue/princess-...,0e691eda369f4e09,5,Fantasy,In stock (16 available),Just as Annie and Liam are busy making plans t...
